# `ingestion/parse.py` — Validation Walkthrough

Steps through each major output of the parser against the FE 501s repair manual.

In [ ]:
import sys
sys.path.insert(0, "..")   # make ingestion package importable from research/

import pypdf
from ingestion.parse import parse_manual, parse_toc

PDF = "../manuals/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du.pdf"

## 1. Table of Contents

`parse_toc` walks the TOC pages and returns every entry as `{number, title, page}`.

In [ ]:
pdf = pypdf.PdfReader(PDF)
toc = parse_toc(pdf)

print(f"Total TOC entries: {len(toc)}")
print("\nFirst 5:")
for e in toc[:5]:
    print(f"  {e['number']:8s}  p.{e['page']:3d}  {e['title']}")
print("\nLast 5:")
for e in toc[-5:]:
    print(f"  {e['number']:8s}  p.{e['page']:3d}  {e['title']}")

## 2. Full parse (no images)

Parse all 378 pages. We skip image extraction here for speed — see section 5 for images.

In [ ]:
pages = parse_manual(PDF, extract_images=False)

print(f"Pages parsed: {len(pages)}")
print(f"Pages with chapter detected: {sum(1 for p in pages if p.chapter_num is not None)}")
print(f"Pages with section heading:  {sum(1 for p in pages if p.section)}")
print(f"Total torque specs:          {sum(len(p.torque_specs) for p in pages)}")
print(f"Total figure refs:           {sum(len(p.figure_refs) for p in pages)}")

## 3. Chapter & section metadata

Spot-check a few content pages to verify chapter/title/section assignment.

In [ ]:
spot_check = [49, 100, 150, 177, 250]  # 0-indexed → pages 50, 101, 151, 178, 251

for i in spot_check:
    p = pages[i]
    print(f"Page {p.page_num:3d} | ch={p.chapter_num:>2}  sec={p.section:<8}  \"{p.chapter_title}\"")

## 4. Torque specs

Show all torque specs from a few chapters to validate extraction quality.

## 5. Inline diagram reference expansion

Diagram callout labels in this manual are typeset directly against the surrounding word with no
separator (e.g. `screws1`, `ofA`, `distanceA`). `_clean_text` expands these to parenthetical form
so the text reads naturally for both humans and the LLM.

Rules:
- Word ending in **lowercase** + **digits** → `screws1` → `screws (1)`
- Word ending in **lowercase** + **1–2 uppercase letters** → `ofA` → `of (A)`
- Bolt specs (`M8`, `M10`) and all-caps words (`WARNING`) are untouched because they end in uppercase.

In [ ]:
p50 = next(p for p in pages if p.page_num == 50)

cases = [
    # (raw_form, expanded_form, description)
    ("ofA",        "of (A)",        "short preposition + label"),
    ("distanceA",  "distance (A)",  "noun + label"),
    ("Remove1",    "Remove (1)",    "verb + callout number"),
    ("screws2",    "screws (2)",    "noun + callout number"),
    ("M10",        "M10",           "bolt spec — must be unchanged"),
    ("WARNING",    "WARNING",       "all-caps word — must be unchanged"),
]

print(f"{'Raw form':<14} {'Expected':<16} {'In text':<16} {'Status'}")
print("-" * 58)
for raw, expanded, desc in cases:
    in_text = expanded in p50.text
    raw_gone = raw not in p50.text if raw != expanded else True
    ok = in_text and raw_gone
    print(f"{raw:<14} {expanded:<16} {str(in_text):<16} {'✓' if ok else '✗'}  {desc}")

In [ ]:
# All torque specs from chapter 7 (Handlebar, controls) and chapter 16 (Brake system)
for ch in [7, 16]:
    ch_pages = [p for p in pages if p.chapter_num == ch]
    specs = [(p.page_num, s) for p in ch_pages for s in p.torque_specs]
    print(f"\nChapter {ch} — {ch_pages[0].chapter_title} ({len(specs)} specs)")
    for page_num, s in specs:
        note = f"  [{s.note}]" if s.note else ""
        print(f"  p.{page_num:3d}  {s.bolt:<8}  {s.nm:6.1f} Nm  ({s.ftlbf} ft·lbf)  {s.description}{note}")

## 5. Images

Parse a single page with image extraction and inspect what comes back.

In [ ]:
from ingestion.parse import ParsedPage, _extract_images
from IPython.display import display, Image as IPImage
import io

# Extract images from page 50 (handlebar chapter — known to have diagrams)
pdf_page = pypdf.PdfReader(PDF).pages[49]
imgs = _extract_images(pdf_page)

print(f"Images on page 50: {len(imgs)}")
for img in imgs:
    print(f"  {img.key:20s}  {img.width}x{img.height}  {len(img.data):,} bytes")

In [ ]:
# Display the largest image from page 50 (most likely the procedure diagram)
largest = max(imgs, key=lambda i: i.width * i.height)
print(f"Displaying: {largest.key}  {largest.width}x{largest.height}")
display(IPImage(data=largest.data))

## 6. Pages with no chapter detected

Front matter and back matter are expected to have no chapter. Verify nothing surprising slipped through.

In [ ]:
no_chapter = [p for p in pages if p.chapter_num is None]
print(f"Pages without chapter: {len(no_chapter)}")
print(f"Page numbers: {[p.page_num for p in no_chapter]}")
print()
# Show what's on a couple of them
for p in no_chapter[:3]:
    print(f"--- Page {p.page_num} ---")
    print(p.text[:300])
    print()